In [ ]:
# Task 2

In [2]:
!pip install -q hazm pandas

In [33]:
import os
import re
import pandas as pd
from collections import Counter

from hazm import Normalizer, word_tokenize
try:
    from hazm import stopwords_list
except ImportError:
    # Different path from older versions
    from hazm.utils import stopwords_list

In [10]:
df = pd.read_csv("D:/my_project/HW_NLP_01_FatemahDadkhah_Task2/asriran.csv")
text_column = "body"

print("Number of rows:", len(df))
df.head(10)

Number of rows: 339834


,title,shortlink,time,service,subgroup,abstract,body
0,پلیس: جرائم خشن و مسلحانه در تهران کاهش یافته است,https://www.asriran.com/003YoB,۰۸:۴۱ - ۲۳ تير ۱۴۰۱,صفحه نخست,عمومی,آمارهای پلیس نشان می‌دهد که جرائم خشن و مسلحان...,رئیس پلیس آگاهی تهران بزرگ اعلام کرد که موضوع ...
1,"وزیر بهداشت:\r\nآغاز اجرای طرح جامع ""دارویار""/...",https://www.asriran.com/003YoC,۰۸:۴۷ - ۲۳ تير ۱۴۰۱,صفحه نخست,اخبار سلامت,هدف اصلی وزارت بهداشت از این طرح این است که پو...,"وزیر بهداشت جزییات طرح ""دارویار"" که اجرای آن آ..."
2,وزارت بهداشت: قیمت دارو برای مصرف کننده ثابت م...,https://www.asriran.com/003YoF,۰۹:۰۸ - ۲۳ تير ۱۴۰۱,صفحه نخست,اجتماعی,NaN,سخنگوی وزارت بهداشت، درمان و آموزش پزشکی در تو...
3,معاون رئیسی: مردم به زودی شاهد اثرات مثبت اقدا...,https://www.asriran.com/003YZP,۱۶:۴۶ - ۱۶ تير ۱۴۰۱,صفحه نخست,سیاسی,معاون امور مجلس رئیس جمهور: استان گیلان دارای ...,معاون رئیس جمهور گفت: دولت برنامه های راهبردی ...
4,دستگیری سارق ۲۰ هزار دلاری ارز دیجیتال در گلستان,https://www.asriran.com/003YZN,۱۶:۴۱ - ۱۶ تير ۱۴۰۱,صفحه نخست,اجتماعی,شهروندان توصیه‌های پلیس فتا را جدی بگیرند و در...,رییس پلیس فتا فرماندهی انتظامی گلستان گفت: سار...
5,رونمایی از آمبولانس حیات وحش استان بوشهر,https://www.asriran.com/003YRS,۱۳:۳۱ - ۱۳ تير ۱۴۰۱,صفحه نخست,محیط زیست,از همین رو پیشنهاد احداث یک واحد درمان و نقاهت...,در آئینی با حضور مدیرعامل صندوق ملی محیط زیست،...
6,معاون وزیر اطلاعات: فضای مجازی بلای خانمان سوز...,https://www.asriran.com/003YoG,۰۹:۲۳ - ۲۳ تير ۱۴۰۱,صفحه نخست,سیاسی,مقام معظم رهبری بیان کردند اگر رهبر نبودند مسئ...,معاون سیاسی، فرهنگی و اجتماعی وزارت اطلاعات گف...
7,آبکش شدن با 60 گلوله پلیس؛ تاوان فرار یک جوان ...,https://www.asriran.com/003YRC,۱۲:۰۶ - ۱۳ تير ۱۴۰۱,صفحه نخست,سیاسی,خانواده این جوان 25 ساله آمریکایی گفته اند رفت...,یک جوان سیاه پوست در شهر آکرون ایالت اوهایو آم...
8,درگذشت دانشجوی دانشگاه فرهنگیان,https://www.asriran.com/003YOn,۱۵:۱۸ - ۱۲ تير ۱۴۰۱,صفحه نخست,حوادث,"""متین حبیب لی"" دانشجومعلم رشته آموزش ابتدایی ۹...",دانشگاه فرهنگیان با صدور پیامی، درگذشت یکی از ...
9,پلیس فتا: تشدید مقابله با لایوهای غیر اخلاقی د...,https://www.asriran.com/003YOj,۱۴:۵۸ - ۱۲ تير ۱۴۰۱,صفحه نخست,اجتماعی,رصد این گونه جریانات غیر اخلاقی که مغایر با شئ...,معاون فرهنگی اجتماعی پلیس فتا فراجا از تشدید ط...


## Removing noise with Regex

In [11]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"  # dingbats
    "\U0001F900-\U0001F9FF"  # supplemental symbols
    "\U00002600-\U000026FF"  # misc symbols
    "]+",
    flags=re.UNICODE,
)

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)")
PUNCTUATION_PATTERN = re.compile(r"[!\"#$%&()*+,./:;<=>?@\[\]^_`{|}~«»–—\-…؟،؛٪]")
DIGIT_PATTERN = re.compile(r"[0-9\u06F0-\u06F9]+") 
MULTI_SPACE_PATTERN = re.compile(r"\s+")


def remove_noise_regex(text: str) -> str:
    """
    Remove the main sources of text noise with Regex:
    links, emojis, numbers, punctuation, and repeated whitespace.
    """
    if not isinstance(text, str):
        return ""

    text = URL_PATTERN.sub(" ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = DIGIT_PATTERN.sub(" ", text)
    text = PUNCTUATION_PATTERN.sub(" ", text)
    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text

sample = df[text_column].iloc[0]
print("Before:", sample)
print("After :", remove_noise_regex(sample))

Before: رئیس پلیس آگاهی تهران بزرگ اعلام کرد که موضوع سرقت مسلحانه از مشاور رئیس فدراسیون کشتی در حال پیگیری است. به گزارش ایسنا، روز دوشنبه خبری مبنی بر «تیراندازی به مشاور علیرضا دبیر و سرقت لندکروز» منتشر و اعلام شد که «علی صفایی، مشاور علیرضا دبیر و مربی سابق تیم ملی کشتی آزاد نوجوانان هدف سرقت مسلحانه در اتوبان امام علی (ع) تهران قرار گرفت و چهارسارق مسلح علاوه بر اینکه خودروی تویوتا لندکروز او را به سرقت بردند، به پای چپ او نیز دو مرتبه شلیک کرده‌اند.» درپی انتشار خبر این حادثه، سرهنگ علی ولیپور گودرزی، رئیس پلیس آگاهی تهران بزرگ در گفت‌وگو با ایسنا، از پیگیری این موضوع از سوی ماموران این پلیس خبرداد و گفت:موضوعی که رخ داده بحث سرقت بوده که همکاران من از همان لحظه اولی که وقوع حادثه به پلیس اطلاع داده شد، در صحنه حاضر شده و موضوع را پیگیری کردند. البته این رویه‌ای است که در مورد تمام جرائم و سرقت‌ها وجود دارد و در حال حاضر نیز پیگیری‌ها و تحقیقات درباره این پرونده از سوی کارآگاهان پلیس آگاهی در حال انجام است. وی در پاسخ به این پرسش که آیا جرائم خشن و مسلحانه در پایتخت افزایش داشت

## Correcting word stress

In [12]:
ELONGATION_PATTERN = re.compile(r"(.)\1{2,}") 


def fix_elongation(text: str) -> str:
    """
    Collapse abnormal letter repetition (elongation) down to a single
    occurrence. Example: 'عالیییی' -> 'عالی'
    """
    return ELONGATION_PATTERN.sub(r"\1", text)

test_word = "عالیییی بووووود خووووب"
print("Before:", test_word)
print("After :", fix_elongation(test_word))

Before: عالیییی بووووود خووووب
After : عالی بود خوب


## Normalization with Hazm

In [21]:
normalizer = Normalizer()

test_text = " جمله      زیبایي روي دیوار حکً شده بود" 
print("Before:", test_text)
print("After :", normalizer.normalize(test_text))

Before:  جمله      زیبایي روي دیوار حکً شده بود
After : جمله زیبایی روی دیوار حک‌شده بود


## Remove stopwords

In [25]:
persian_stopwords = set(stopwords_list())

NEGATION_WORDS = {
    "نه", "نیست", "نمی", "نمیشود", "نمی‌شود", "نبود", "نداشت",
    "هیچ", "بدون", "نکرد", "نکردم", "نداره", "ندارد", "نداریم",
    "نکنید", "نمی‌کند", "نمیکند", "نمی‌کنم", "نمیکنم",
}

custom_stopwords = persian_stopwords - NEGATION_WORDS

print(f"Total standard Hazm stopwords: {len(persian_stopwords)}")
print(f"Stopwords remaining after excluding negation words: {len(custom_stopwords)}")
print(f"Sample of negation words that were kept: {NEGATION_WORDS & persian_stopwords}")


def remove_stopwords(text: str) -> str:
    """
    Tokenize the text with Hazm and remove stopwords, except negation words.
    """
    tokens = word_tokenize(text)
    filtered_tokens = [tok for tok in tokens if tok not in custom_stopwords]
    return " ".join(filtered_tokens)


test_sentence = "این محصول اصلا خوب نبود، کیفیتش خیلی پایین است و اصلا راضی نیستم، نه به کسی پیشنهادش نمی‌کنم"
print()
print("Before:", test_sentence)
print("After :", remove_stopwords(test_sentence))
print("--> The word 'نیست' (is not) was preserved, so the negative meaning was not lost.")

Total standard Hazm stopwords: 389
Stopwords remaining after excluding negation words: 382
Sample of negation words that were kept: {'نبود', 'ندارد', 'نمی\u200cشود', 'بدون', 'نیست', 'نه', 'هیچ'}

Before: این محصول اصلا خوب نبود، کیفیتش خیلی پایین است و اصلا راضی نیستم، نه به کسی پیشنهادش نمی‌کنم
After : محصول اصلا نبود ، کیفیتش پایین اصلا راضی نیستم ، نه پیشنهادش نمی‌کنم
--> The word 'نیست' (is not) was preserved, so the negative meaning was not lost.


In [ ]:
def clean_persian_text(text: str) -> str:
    """
    Full Persian text preprocessing pipeline.

    Steps, in order:
        1. Remove noise (links, emojis, numbers, punctuation) with Regex
        2. Fix letter elongation with Regex
        3. Normalize with Hazm's Normalizer (unify Arabic/Persian chars, half-spaces)
        4. Tokenize and remove stopwords (keeping negation words)

    Parameters
    ----------
    text : str
        Raw input text (Persian)

    Returns
    -------
    str
        Cleaned text, ready for use in downstream NLP models
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # Step 1: noise removal
    text = remove_noise_regex(text)

    # Step 2: fix letter elongation
    text = fix_elongation(text)

    # Step 3: normalize with Hazm
    text = normalizer.normalize(text)

    # Step 4: tokenize and remove stopwords (keeping negations)
    text = remove_stopwords(text)

    return text

raw_sample = "سلام دوستان، این محصول واقعا عالیییی بود و من خیلی راضی هستم!!! پیشنهاد میکنم حتما بخرید 😍😍😍"
print("Raw text:")
print(raw_sample)
print()
print("Cleaned text:")
print(clean_persian_text(raw_sample))

Raw text:
سلام دوستان، این محصول واقعا عالیییی بود و من خیلی راضی هستم!!! پیشنهاد میکنم حتما بخرید 😍😍😍

Cleaned text:
سلام دوستان محصول واقعا راضی هستم پیشنهاد حتما بخرید


In [30]:
df["clean_text"] = df[text_column].apply(clean_persian_text)

df[[text_column, "clean_text"]].head(10)

,body,clean_text
0,رئیس پلیس آگاهی تهران بزرگ اعلام کرد که موضوع ...,رئیس پلیس آگاهی تهران اعلام موضوع سرقت مسلحانه...
1,"وزیر بهداشت جزییات طرح ""دارویار"" که اجرای آن آ...",وزیر بهداشت جزییات طرح دارویار اجرای آغاز تشری...
2,سخنگوی وزارت بهداشت، درمان و آموزش پزشکی در تو...,سخنگوی وزارت بهداشت درمان آموزش پزشکی توئیتی ت...
3,معاون رئیس جمهور گفت: دولت برنامه های راهبردی ...,معاون رئیس‌جمهور دولت برنامه‌های راهبردی اساسی...
4,رییس پلیس فتا فرماندهی انتظامی گلستان گفت: سار...,رییس پلیس فتا فرماندهی انتظامی گلستان سارقی تو...
5,در آئینی با حضور مدیرعامل صندوق ملی محیط زیست،...,آئینی حضور مدیرعامل صندوق ملی محیط‌زیست آمبولا...
6,معاون سیاسی، فرهنگی و اجتماعی وزارت اطلاعات گف...,معاون سیاسی فرهنگی اجتماعی وزارت اطلاعات برجام...
7,یک جوان سیاه پوست در شهر آکرون ایالت اوهایو آم...,جوان سیاه‌پوست شهر آکرون ایالت اوهایو آمریکا خ...
8,دانشگاه فرهنگیان با صدور پیامی، درگذشت یکی از ...,دانشگاه فرهنگیان صدور پیامی درگذشت دانشجویان د...
9,معاون فرهنگی اجتماعی پلیس فتا فراجا از تشدید ط...,معاون فرهنگی اجتماعی پلیس فتا فراجا تشدید طرح ...


## Final analysis

In [31]:
all_words = []
for text in df["clean_text"]:
    all_words.extend(text.split())

word_freq = Counter(all_words)
top_10_words = word_freq.most_common(10)

print("Top 10 most frequent words in the cleaned DataFrame:")
for rank, (word, freq) in enumerate(top_10_words, start=1):
    print(f"{rank:2d}. {word:15s} -> {freq} occurrences")

common_stopwords_check = {"از", "به", "که", "را", "این", "با", "در"}
leaked = [w for w, _ in top_10_words if w in common_stopwords_check]
print()
if leaked:
    print(f"WARNING: these common stopwords still appear in the top-10 list: {leaked}")
    print("The pipeline needs review.")
else:
    print("SUCCESS: none of the common stopwords (از/به/که/را/...) appear in the top-10 list.")
    print("The preprocessing pipeline is working correctly.")

Top 10 most frequent words in the cleaned DataFrame:
 1. کشور            -> 348804 occurrences
 2. ایران           -> 343515 occurrences
 3. سال             -> 305374 occurrences
 4. گزارش           -> 253460 occurrences
 5. قرار            -> 215833 occurrences
 6. مردم            -> 199975 occurrences
 7. روز             -> 185696 occurrences
 8. آمریکا          -> 171320 occurrences
 9. ادامه           -> 168703 occurrences
10. کار             -> 167774 occurrences

SUCCESS: none of the common stopwords (از/به/که/را/...) appear in the top-10 list.
The preprocessing pipeline is working correctly.


In [36]:
output_path = "D:/my_project/HW_NLP_01_FatemahDadkhah_Task2/cleaned_persian_texts.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Output file saved: {output_path}")
print(f"Total rows: {len(df)}")

Output file saved: D:/my_project/HW_NLP_01_FatemahDadkhah_Task2/cleaned_persian_texts.csv
Total rows: 339834
